# ODE Solver Demo

Comparing Euler, RK4, and adaptive solvers on standard test problems.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.pardir, "src"))

import numpy as np
import matplotlib.pyplot as plt

from nms.ode import euler_solve, rk4_solve, adaptive_solve
from nms.analysis import convergence_rate, stability_region, stability_function_euler, stability_function_rk4

## 1. Exponential Decay: dy/dt = -y

In [ ]:
def exp_decay(t, y):
    return -y

t_span = (0, 5)
y0 = [1.0]

t_euler, y_euler = euler_solve(exp_decay, t_span, y0, h=0.2)
t_rk4, y_rk4 = rk4_solve(exp_decay, t_span, y0, h=0.2)
t_adap, y_adap = adaptive_solve(exp_decay, t_span, y0, tol=1e-8)

t_exact = np.linspace(*t_span, 200)
y_exact = np.exp(-t_exact)

plt.figure(figsize=(10, 5))
plt.plot(t_exact, y_exact, "k-", lw=2, label="Exact")
plt.plot(t_euler, y_euler[:, 0], "o--", ms=3, label=f"Euler (h=0.2, {len(t_euler)} pts)")
plt.plot(t_rk4, y_rk4[:, 0], "s--", ms=3, label=f"RK4 (h=0.2, {len(t_rk4)} pts)")
plt.plot(t_adap, y_adap[:, 0], "^--", ms=3, label=f"Adaptive ({len(t_adap)} pts)")
plt.xlabel("t")
plt.ylabel("y")
plt.title("Exponential Decay: dy/dt = −y")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Convergence Study

In [ ]:
step_sizes = [0.5, 0.25, 0.125, 0.0625, 0.03125]
euler_errors = []
rk4_errors = []

for h in step_sizes:
    _, ye = euler_solve(exp_decay, (0, 1), [1.0], h=h)
    _, yr = rk4_solve(exp_decay, (0, 1), [1.0], h=h)
    exact_final = np.exp(-1.0)
    euler_errors.append(abs(ye[-1, 0] - exact_final))
    rk4_errors.append(abs(yr[-1, 0] - exact_final))

euler_rates = convergence_rate(euler_errors, step_sizes)
rk4_rates = convergence_rate(rk4_errors, step_sizes)

print(f"Euler convergence rates: {euler_rates}  (expected ~1)")
print(f"RK4   convergence rates: {rk4_rates}  (expected ~4)")

plt.figure(figsize=(8, 5))
plt.loglog(step_sizes, euler_errors, "o-", label="Euler")
plt.loglog(step_sizes, rk4_errors, "s-", label="RK4")
plt.loglog(step_sizes, [h**1 * euler_errors[0] / step_sizes[0]**1 for h in step_sizes], "k--", alpha=0.4, label="O(h)")
plt.loglog(step_sizes, [h**4 * rk4_errors[0] / step_sizes[0]**4 for h in step_sizes], "k:", alpha=0.4, label="O(h⁴)")
plt.xlabel("Step size h")
plt.ylabel("Error at t = 1")
plt.title("Convergence Study")
plt.legend()
plt.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.show()

## 3. Stability Regions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, R, name in [
    (axes[0], stability_function_euler, "Forward Euler"),
    (axes[1], stability_function_rk4, "RK4"),
]:
    region = stability_region(R)
    ax.contourf(region["X"], region["Y"], region["is_stable"].astype(float),
                levels=[0.5, 1.5], colors=["#3b82f6"], alpha=0.4)
    ax.contour(region["X"], region["Y"], region["magnitude"],
               levels=[1.0], colors=["#1e40af"], linewidths=2)
    ax.set_xlabel("Re(z)")
    ax.set_ylabel("Im(z)")
    ax.set_title(f"Stability Region: {name}")
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()